# 171. LLM 评测去污染：Exact、N-gram、MinHash 与 Clean Score 怎样设计？

> **面试问题：怎样判断训练语料是否包含 benchmark？精确/近似/改写污染如何检测，清洗前后分数怎样报告？**

## 先给结论

去污染不是一次字符串查重。应保留原文证据，分层使用规范化 exact hash、长 n-gram containment、近重复候选与人工/语义复核；阈值在带标签样本上校准，并按 benchmark family 与时间切分。最终同时报告 raw、clean、被移除比例和不确定样本，避免清洗规则选择性美化结果。

## 推荐回答主线

1. 先定义 contamination unit：题面、答案、解释、模板族还是整页，并保留来源/时间/provenance。
2. exact hash 处理直接复制，n-gram containment 处理长片段嵌入，MinHash 只做近重复候选生成。
3. 在正负对上校准阈值，特别检查短文本、样板话、翻译和 paraphrase 的误报/漏报。
4. 训练前删除/隔离，评测时输出 raw-clean gap、slice、置信区间、规则版本和审计清单。

## 教学边界

示例用中英文小字符串与标准库散列，不扫描真实训练集，也不声称字符相似能发现语义改写。生产需分布式候选生成、语言专用规范化、隐私控制与人工复核。

## 一手资料

- [Deduplicating Training Data Makes Language Models Better](https://arxiv.org/abs/2107.06499)
- [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783)
- [DataComp-LM](https://arxiv.org/abs/2406.11794)


In [ ]:
import hashlib  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。
import unicodedata  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。

# 训练片段包含直接复制、包裹复制、近似改写和无关内容。
benchmark = [  # 计算并保存当前步骤的中间状态。
    {"id": "math-1", "family": "fractions", "text": "What is 3/4 plus 5/6? Answer: 19/12"},  # 执行当前语句以推进本节示例。
    {"id": "cn-1", "family": "capital", "text": "中国的首都是哪里？答案：北京"},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
training = [  # 计算并保存当前步骤的中间状态。
    "WHAT IS 3/4 PLUS 5/6?   ANSWER: 19/12",  # 执行当前语句以推进本节示例。
    "练习册第七页：中国的首都是哪里？答案：北京。下一题开始。",  # 执行当前语句以推进本节示例。
    "把四分之三与六分之五相加，结果为十二分之十九。",  # 执行当前语句以推进本节示例。
    "今天北京天气晴朗，与问答评测无关。",  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。

assert len(benchmark) == 2  # 用受控断言验证关键不变量。
assert len(training) == 4  # 用受控断言验证关键不变量。
assert len({item["id"] for item in benchmark}) == len(benchmark)  # 用受控断言验证关键不变量。


## 1. 可审计规范化：Unicode、大小写和空白，不删除语义符号

规范化用于减少表面差异，但规则越激进越可能把不同题合并。NFKC、casefold、空白折叠和首尾标点是常见起点；分数斜杠、负号、代码缩进等可能承载语义，应按领域保留。


In [ ]:
def normalize_text(text):  # 定义本节可复用的核心函数。
    text = unicodedata.normalize("NFKC", text).casefold()  # 计算并保存当前步骤的中间状态。
    text = re.sub(r"\s+", " ", text).strip()  # 计算并保存当前步骤的中间状态。
    return text  # 返回当前分支计算出的结果。

# 大小写/多空白被统一，分数字符仍保留，中文不被删除。
normalized_benchmark = [normalize_text(item["text"]) for item in benchmark]  # 计算并保存当前步骤的中间状态。
normalized_training = [normalize_text(text) for text in training]  # 计算并保存当前步骤的中间状态。
assert normalized_training[0] == normalized_benchmark[0]  # 用受控断言验证关键不变量。
assert "3/4" in normalized_training[0]  # 用受控断言验证关键不变量。
assert "中国" in normalized_training[1]  # 用受控断言验证关键不变量。


## 2. Exact hash：快速发现直接复制，但碰撞与规则版本要记录

对规范化文本做 SHA-256 可低成本查 exact overlap。散列命中后仍应回看原文/provenance；规范化版本必须绑定摘要，因为规则变化会改变命中集合。hash 不会发现文档中嵌入的子串。


In [ ]:
def exact_digest(text, normalizer_version="nfkc-casefold-space-v1"):  # 定义本节可复用的核心函数。
    payload = (normalizer_version + "\0" + normalize_text(text)).encode("utf-8")  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload).hexdigest()  # 返回当前分支计算出的结果。

# 直接复制命中；包裹了上下文的复制不再 exact；规则版本变化会改变摘要。
benchmark_hashes = {exact_digest(item["text"]): item["id"] for item in benchmark}  # 计算并保存当前步骤的中间状态。
exact_hits = [benchmark_hashes.get(exact_digest(text)) for text in training]  # 计算并保存当前步骤的中间状态。
assert exact_hits[0] == "math-1"  # 用受控断言验证关键不变量。
assert exact_hits[1] is None  # 用受控断言验证关键不变量。
assert exact_digest(training[0], "v2") != exact_digest(training[0], "v1")  # 用受控断言验证关键不变量。


## 3. N-gram containment：检测 benchmark 被嵌入较长训练文档

Jaccard 对“短 benchmark 包在长页面”会被大量额外文本稀释，因此去污染常看 benchmark n-gram 有多少被 training 覆盖。n 越大越精确但怕小改写；短题需降低 n 或人工规则。


In [ ]:
def char_ngrams(text, n):  # 定义本节可复用的核心函数。
    compact = re.sub(r"\s+", "", normalize_text(text))  # 计算并保存当前步骤的中间状态。
    return {compact[i:i + n] for i in range(max(0, len(compact) - n + 1))}  # 返回当前分支计算出的结果。

def containment(short_text, long_text, n=5):  # 定义本节可复用的核心函数。
    short, long = char_ngrams(short_text, n), char_ngrams(long_text, n)  # 计算并保存当前步骤的中间状态。
    return len(short & long) / len(short) if short else 0.0  # 返回当前分支计算出的结果。

# 包裹复制应高 containment；无关天气文本明显更低；得分范围合法。
wrapped_score = containment(benchmark[1]["text"], training[1], n=4)  # 计算并保存当前步骤的中间状态。
unrelated_score = containment(benchmark[1]["text"], training[3], n=4)  # 计算并保存当前步骤的中间状态。
assert wrapped_score > 0.9  # 用受控断言验证关键不变量。
assert wrapped_score > unrelated_score  # 用受控断言验证关键不变量。
assert 0 <= unrelated_score <= 1  # 用受控断言验证关键不变量。


## 4. 最长公共连续 token：给审计员展示命中证据片段

只有一个相似度分数很难审计。动态规划可找最长公共连续 token 片段，输出长度和文本证据。生产大规模不应用 O(nm) 全对全；先用 hash/LSH 生成候选，再对少量 pair 精查。


In [ ]:
def simple_tokens(text):  # 定义本节可复用的核心函数。
    return re.findall(r"[a-z0-9]+(?:/[0-9]+)?|[\u4e00-\u9fff]", normalize_text(text))  # 返回当前分支计算出的结果。

def longest_common_run(a, b):  # 定义本节可复用的核心函数。
    ta, tb = simple_tokens(a), simple_tokens(b)  # 计算并保存当前步骤的中间状态。
    dp = [0] * (len(tb) + 1)  # 计算并保存当前步骤的中间状态。
    best_len, best_end = 0, 0  # 计算并保存当前步骤的中间状态。
    for i, token_a in enumerate(ta, 1):  # 遍历输入元素以累积或检查结果。
        new = [0] * (len(tb) + 1)  # 计算并保存当前步骤的中间状态。
        for j, token_b in enumerate(tb, 1):  # 遍历输入元素以累积或检查结果。
            if token_a == token_b:  # 按当前条件选择后续控制路径。
                new[j] = dp[j - 1] + 1  # 计算并保存当前步骤的中间状态。
                if new[j] > best_len:  # 按当前条件选择后续控制路径。
                    best_len, best_end = new[j], i  # 计算并保存当前步骤的中间状态。
        dp = new  # 计算并保存当前步骤的中间状态。
    return ta[best_end - best_len:best_end]  # 返回当前分支计算出的结果。

# exact 题应给出完整 token run；无关文本不会伪造同样长的连续证据。
run_exact = longest_common_run(benchmark[0]["text"], training[0])  # 计算并保存当前步骤的中间状态。
run_unrelated = longest_common_run(benchmark[0]["text"], training[3])  # 计算并保存当前步骤的中间状态。
assert len(run_exact) >= 7  # 用受控断言验证关键不变量。
assert len(run_exact) > len(run_unrelated)  # 用受控断言验证关键不变量。
assert "19/12" in run_exact  # 用受控断言验证关键不变量。


## 5. MinHash 只做候选：近似 Jaccard 需要最终精排

对 shingle 集合取多个随机置换下的最小 hash，签名相等比例估计 Jaccard。这里用确定的 keyed hash；估计有方差，不能直接作为删除判决。LSH banding 可避免全对全。


In [ ]:
def minhash_signature(shingles, permutations=128):  # 定义本节可复用的核心函数。
    signature = []  # 计算并保存当前步骤的中间状态。
    for seed in range(permutations):  # 遍历输入元素以累积或检查结果。
        values = [int.from_bytes(hashlib.blake2b(item.encode(), digest_size=8, key=seed.to_bytes(4, "little")).digest(), "little") for item in shingles]  # 计算并保存当前步骤的中间状态。
        signature.append(min(values) if values else 2**64 - 1)  # 执行当前语句以推进本节示例。
    return np.array(signature, dtype=np.uint64)  # 返回当前分支计算出的结果。

def signature_similarity(sig_a, sig_b):  # 定义本节可复用的核心函数。
    return float(np.mean(sig_a == sig_b))  # 返回当前分支计算出的结果。

# 相同集合估计为 1；包裹复制比无关文本更相似；签名长度固定。
bench_set = char_ngrams(benchmark[1]["text"], 3)  # 计算并保存当前步骤的中间状态。
sig_b = minhash_signature(bench_set)  # 计算并保存当前步骤的中间状态。
sig_wrapped = minhash_signature(char_ngrams(training[1], 3))  # 计算并保存当前步骤的中间状态。
sig_unrelated = minhash_signature(char_ngrams(training[3], 3))  # 计算并保存当前步骤的中间状态。
assert signature_similarity(sig_b, sig_b) == 1.0  # 用受控断言验证关键不变量。
assert signature_similarity(sig_b, sig_wrapped) > signature_similarity(sig_b, sig_unrelated)  # 用受控断言验证关键不变量。
assert len(sig_b) == 128  # 用受控断言验证关键不变量。


## 6. 阈值校准：最大化目标不是“删得越多越好”

用人工标注 candidate pair 选择阈值，显式计算 precision/recall/F1 和 false-positive 成本。模板化短语可能高相似却不泄露答案；paraphrase 则可能低字符相似却实质污染，需要单独语义/翻译审计。


In [ ]:
def binary_metrics(scores, labels, threshold):  # 定义本节可复用的核心函数。
    pred = np.asarray(scores) >= threshold  # 计算并保存当前步骤的中间状态。
    labels = np.asarray(labels, dtype=bool)  # 计算并保存当前步骤的中间状态。
    tp, fp, fn = (pred & labels).sum(), (pred & ~labels).sum(), (~pred & labels).sum()  # 计算并保存当前步骤的中间状态。
    precision = tp / max(tp + fp, 1)  # 计算并保存当前步骤的中间状态。
    recall = tp / max(tp + fn, 1)  # 计算并保存当前步骤的中间状态。
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)  # 计算并保存当前步骤的中间状态。
    return precision, recall, f1  # 返回当前分支计算出的结果。

# 在受控标注上搜索 F1；最佳阈值来自候选集合，指标在 [0,1]。
cal_scores = [1.0, 0.96, 0.81, 0.55, 0.21, 0.05]  # 计算并保存当前步骤的中间状态。
cal_labels = [1, 1, 1, 0, 0, 0]  # 计算并保存当前步骤的中间状态。
thresholds = np.linspace(0, 1, 21)  # 计算并保存当前步骤的中间状态。
metrics = [binary_metrics(cal_scores, cal_labels, threshold) for threshold in thresholds]  # 计算并保存当前步骤的中间状态。
best_index = int(np.argmax([item[2] for item in metrics]))  # 计算并保存当前步骤的中间状态。
best_threshold = float(thresholds[best_index])  # 计算并保存当前步骤的中间状态。
assert best_threshold in thresholds  # 用受控断言验证关键不变量。
assert all(0 <= value <= 1 for value in metrics[best_index])  # 用受控断言验证关键不变量。
assert metrics[best_index][2] == 1.0  # 用受控断言验证关键不变量。


## 7. Family 与时间合同：字符串不同也可能来自同一题模板

训练/评测应按 benchmark family、来源和时间整体隔离。发布后的网页可能反向进入后续预训练语料；temporal cutoff 与 crawl timestamp 必须可追踪。无法确定的候选进入 quarantine，不静默当 clean。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AuditDecision:  # 定义承载本节状态与行为的数据结构。
    train_doc_id: str  # 执行当前语句以推进本节示例。
    benchmark_id: str  # 执行当前语句以推进本节示例。
    rule: str  # 执行当前语句以推进本节示例。
    score: float  # 执行当前语句以推进本节示例。
    action: str  # 执行当前语句以推进本节示例。
    rule_version: str  # 执行当前语句以推进本节示例。

def temporal_ok(crawl_day, benchmark_release_day):  # 定义本节可复用的核心函数。
    return crawl_day < benchmark_release_day  # 返回当前分支计算出的结果。

# benchmark 发布后的抓取不满足时间隔离；决策必须绑定规则与动作。
decision = AuditDecision("train-002", "cn-1", "char4-containment", wrapped_score, "quarantine", "decon-v3")  # 计算并保存当前步骤的中间状态。
assert not temporal_ok("2025-07-01", "2025-01-15")  # 用受控断言验证关键不变量。
assert temporal_ok("2024-12-01", "2025-01-15")  # 用受控断言验证关键不变量。
assert decision.action in {"keep", "remove", "quarantine"}  # 用受控断言验证关键不变量。


## 8. Clean score 报告：同时给覆盖率、raw-clean gap 与不确定性

清洗后分数只在 clean subset 上计算，样本构成已变化，因此要同时给 raw、clean、污染率和分 slice 结果。若被标污染样本更容易，raw-clean gap 能提示虚高，但不是污染因果效应的完整证明。


In [ ]:
def raw_clean_report(correct, contaminated):  # 定义本节可复用的核心函数。
    correct = np.asarray(correct, dtype=float)  # 计算并保存当前步骤的中间状态。
    contaminated = np.asarray(contaminated, dtype=bool)  # 计算并保存当前步骤的中间状态。
    clean = ~contaminated  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "raw_accuracy": float(correct.mean()),  # 执行当前语句以推进本节示例。
        "clean_accuracy": float(correct[clean].mean()) if clean.any() else math.nan,  # 执行当前语句以推进本节示例。
        "contamination_rate": float(contaminated.mean()),  # 执行当前语句以推进本节示例。
        "removed": int(contaminated.sum()),  # 执行当前语句以推进本节示例。
        "kept": int(clean.sum()),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

# 人工示例中污染样本全对，raw 分数因此高于 clean；计数守恒。
report = raw_clean_report([1, 1, 1, 0, 0, 1], [1, 1, 0, 0, 0, 0])  # 计算并保存当前步骤的中间状态。
assert report["raw_accuracy"] > report["clean_accuracy"]  # 用受控断言验证关键不变量。
assert report["removed"] + report["kept"] == 6  # 用受控断言验证关键不变量。
assert 0 <= report["contamination_rate"] <= 1  # 用受控断言验证关键不变量。


## 面试收束与生产替换点

完整回答不要停在算法名：先说清业务目标、输入输出与信任边界，再给核心数据结构/公式和可执行 oracle，最后落到离线切片、线上 SLO、成本、安全、版本、灰度与回滚。这里的受控实现用于解释机制和发现反例；真实模型编码器、分布式索引、协议 SDK、安全沙箱、监控与持久层应作为可替换组件，并用同一合同验收。

典型追问包括：数据规模扩大后瓶颈在哪？近似步骤损失了什么？哪个状态必须持久化？超时或部分失败怎样降级？版本错配为何不能静默兼容？离线指标上升是否来自污染、权限泄漏、评测器偏差或重复样本？
